# Notebook 001. Public data download
-------
Use this notebook to download the public datasets used in the study. Each dataset has a dedicated cell that can be run in isolation.

This notebook downloads the raw data only - data pre-processing occurs in notebooks 002 and 003. One exception is the Natura 2000 vectors, which are processed to extract the AOI so that AOI-dependent download cells below can run.

Three of the data sources require credentials, all supplied through a local `.env` file stored in the repository root directory (copy `env.example` to `.env`; see
README.md for more details). Set the following values in `.env`:

- `EE_PROJECT`: requires a Google Earth Engine Cloud project ID to download AlphaEarth embeddings. Authenticate the machine once beforehand by running
`earthengine authenticate`.

- `CLMS_SERVICE_KEY_PATH`: requires a Copernicus Land Monitoring Service (CLMS) service-key JSON for CORINE Land Cover (see: https://land.copernicus.eu/en/how-to-guides/how-to-download-spatial-data/how-to-create-api-tokens).

- `WEKEO_USERNAME` and `WEKEO_PASSWORD`: requires WEkEO account credentials for Copernicus HR-VPP (see: registration at https://www.wekeo.eu).
  
Please refer to each dataset's original source for licensing and usage details.

## Automated downloads (using cells in this notebook)

| Name | Year | Data Type(s) | Purpose |
|---|---|---|---|
| Natura 2000 (European Environment Agency) | 2024 | Vector (protected-area boundaries) | Reference labels |
| OpenStreetMap | 2026 | Vector (paved roads, unpaved roads, footpaths) | Reference labels<br>Baseline feature set |
| European Forest Disturbance Atlas | 1985-2023 | 39-band raster (annual forest disturbances, 1985-2023; 10 m) | Reference labels |
| CORINE Land Cover | 2018 | Vector (forest classes: broadleaf / coniferous / mixed) | Reference labels<br>Stratified analysis |
| Forest and Buildings removed Copernicus DEM (FABDEM) | 2022 | 1-band raster, elevation (slope, aspect, roughness derived; ~25 m) | Baseline feature set<br>Stratified analysis |
| ESA WorldCover Annual Composites | 2020 | 12-band raster (Sentinel-2 optical, NDVI percentiles, Sentinel-1 radar, SWIR; 10-30 m) | Conventional Earth observation feature set |
| Copernicus Vegetation Phenology and Productivity (VPP) | 2020 | 10x 1-band rasters (vegetation phenology and productivity metrics; 10 m) | Conventional Earth observation feature set |
| AlphaEarth Foundations Embeddings | 2020 | 64-band raster (foundation-model embeddings; 10 m) | AlphaEarth feature set |
| TESSERA Embeddings (v1.1 or v2, chosen in the TESSERA cell) | 2020 | 128-band raster (foundation-model embeddings; 10 m) | TESSERA feature set |
| Sabatini Primary Forest Probability Map (manual download) | 2020 | 3x 1-band rasters (primary-forest probability; 250m) | Existing study evaluation |
| Munteanu et al. Forest Structure and HCVF Maps | 2022 | 4x 1-band rasters (structural complexity, HCVF; 30 m) | Existing study evaluation |
| Kathmann et al. Potential Primary Forests Map of Romania | 2017 | Vector (potential primary-forest polygons) | Existing study evaluation |

## Manual downloads

- **Forest parcel map, management plans, virgin/quasi-virgin forest and ownership layers:** Download ogf_labels_predictions.gpkg from https://doi.org/10.5281/zenodo.22693148 and place it in `data/processed/vectors/labels/`. Then run the notebooks in the normal order. Notebook 003 sees that the forest records are missing, skips the cells that need them and rebuilds the parcel map and the reference labels from the published file. Skip the reference label construction notebook 005. The study's original forest management datasets were compiled by Fundația Conservation Carpathia, who must be contacted to access. The current management plans are published on Romania's Ministry of Environment, Waters and Forests GIS portal (https://mmediu.ro/en/portal-gis).
- **Sabatini (2020) primary forest map:** the iDiv repository (https://idata.idiv.de/ddm/Data/ShowData/1841) needs a browser session, so download the files manually, place them in `data/raw/existing_products/sabatini/` and run the provenance cell below.
- **Schickhofer & Schwarz primary forest inventory:** not a public download; request it from the authors. Source: Schickhofer, M. and Schwarz, U. (2019), Inventory of Potential Primary and Old-Growth Forest Areas in Romania (PRIMOFARO). Report commissioned by EuroNatur Foundation.
- **Aerial orthomosaic of the study area (basemap of the Figure 1 map panels)**: not public; notebook 012 uses OpenStreetMap tiles when it is absent.

## Natura 2000 (EEA, end 2024)

Downloads the EU-wide Natura 2000 spatial dataset (OGC GeoPackage) to
`data/raw/vectors/natura_2000/`. The two study sites (Munții Făgăraș; Râul Târgului - Argeșel - Râușor)
are selected by SITECODE from the Natura 2000 site polygons and written in the
project CRS (3035).

Source: European Environment Agency, *Natura 2000 data - the European network
of protected sites*, version end 2024 (https://www.eea.europa.eu/en/datahub/datahubitem-view/6fc8ad2d-195d-40f4-bdec-576e7d1268e4)

In [ ]:
# Natura 2000 (EEA, end 2024). Downloads the EU-wide boundaries to data/raw/.
# The two study sites are extracted later, in the processing step, not here.
import geopandas as gpd
import requests

from utils.paths import get_project_paths
from utils.provenance import (
    format_provenance_result,
    provenance_path_for,
    verify_provenance,
    write_provenance,
)
from utils.terminology import CRS

# Resolve paths from the repo root, so this works whatever folder the notebook runs from.
paths = get_project_paths()
OUT_DIR = paths.raw / "vectors" / "natura_2000"
OUT_FILE = OUT_DIR / "natura2000_end2024.gpkg"

# Download link from the EEA datahub (Natura 2000 - Spatial data, OGC GeoPackage).
DATA_URL = "https://sdi.eea.europa.eu/datashare/s/mwzs9eNsJ9Sn4Q4/download?path=%2F&files=Natura2000_end2024.gpkg"

# Set to True to download again even if the file already exists.
FORCE = False

OUT_DIR.mkdir(parents=True, exist_ok=True)

if OUT_FILE.exists() and not FORCE:
    print(f"Already downloaded: {OUT_FILE}")
else:
    print(f"Downloading to {OUT_FILE} ...")
    with requests.get(DATA_URL, stream=True, timeout=300) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        done = 0
        with open(OUT_FILE, "wb") as f:
            # Saved in 1 MB pieces so the whole file is never held in memory at once.
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                done += len(chunk)
                if total:
                    print(f"\r  {done/1e6:7.1f} / {total/1e6:7.1f} MB", end="")
        print()
    print("Done.")

# Records the file's fingerprint on first download; on later runs
# checks it matches the study's version.
prov = provenance_path_for(OUT_FILE)
if prov.exists():
    print(format_provenance_result(verify_provenance(OUT_FILE)))
else:
    write_provenance(OUT_FILE, source=DATA_URL, extra={"dataset": "Natura 2000 (EEA), end 2024"})
    print(format_provenance_result(None, recorded=True))


AOI_FILE = paths.aoi
AOI_SITECODES = ("ROSAC0122", "ROSAC0381")  # Munții Făgăraș; Râul Târgului - Argeșel - Râușor

if AOI_FILE.exists():
    print(f"AOI present: {AOI_FILE.name}")
else:
    sites = gpd.read_file(OUT_FILE, layer="naturasite_polygon")
    aoi = sites[sites["SITECODE"].isin(AOI_SITECODES)].to_crs(CRS)
    if len(aoi) != len(AOI_SITECODES):
        raise RuntimeError(
            f"Expected {len(AOI_SITECODES)} sites, found {len(aoi)}: {sorted(aoi['SITECODE'])}."
        )
    AOI_FILE.parent.mkdir(parents=True, exist_ok=True)
    aoi.to_file(AOI_FILE)
    print(f"Wrote AOI ({len(aoi)} sites) to {AOI_FILE.name}")

## OpenStreetMap (Geofabrik Romania, GeoPackage)

Downloads the Romania OpenStreetMap GeoPackage (`.gpkg.zip`) to
`data/raw/vectors/open_street_map/` and extracts the `.gpkg`.

Source: OpenStreetMap contributors, via Geofabrik
(https://download.geofabrik.de/europe/romania.html).

In [ ]:
# OpenStreetMap (Geofabrik Romania GeoPackage). Downloads the .gpkg.zip to data/raw/,
# extracts the .gpkg, and removes the archive. Classification and AOI clipping are
# done later, in the processing step.
import zipfile

import requests

from utils.paths import get_project_paths
from utils.provenance import (
    format_provenance_result,
    provenance_path_for,
    verify_provenance,
    write_provenance,
)

# Resolve paths from the repo root, so this works whatever folder the notebook runs from.
paths = get_project_paths()
OUT_DIR = paths.raw / "vectors" / "open_street_map"
ZIP_FILE = OUT_DIR / "romania-latest-free.gpkg.zip"

# Geofabrik Romania GeoPackage (canonical page: download.geofabrik.de/europe/romania.html).
DATA_URL = "https://download.geofabrik.de/europe/romania-latest-free.gpkg.zip"

# Set to True to download and extract again even if the .gpkg already exists.
FORCE = False

OUT_DIR.mkdir(parents=True, exist_ok=True)

existing = sorted(OUT_DIR.rglob("*.gpkg"))
if existing and not FORCE:
    OUT_FILE = existing[0]
    print(f"Already downloaded and extracted: {OUT_FILE}")
else:
    print(f"Downloading to {ZIP_FILE} ...")
    with requests.get(DATA_URL, stream=True, timeout=300) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        done = 0
        with open(ZIP_FILE, "wb") as f:
            # Saved in 1 MB pieces so the whole archive is never held in memory at once.
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                done += len(chunk)
                if total:
                    print(f"\r  {done/1e6:7.1f} / {total/1e6:7.1f} MB", end="")
        print()

    print("Extracting ...")
    with zipfile.ZipFile(ZIP_FILE) as z:
        members = [m for m in z.namelist() if m.lower().endswith(".gpkg")]
        if not members:
            raise RuntimeError("No .gpkg found inside the Geofabrik archive.")
        z.extract(members[0], OUT_DIR)
    OUT_FILE = OUT_DIR / members[0]
    ZIP_FILE.unlink()  # keep only the extracted .gpkg
    print(f"Extracted: {OUT_FILE}")

# Records the file's fingerprint on first download; on later runs
# checks it matches the study's version.
prov = provenance_path_for(OUT_FILE)
if prov.exists():
    print(format_provenance_result(verify_provenance(OUT_FILE)))
else:
    write_provenance(
        OUT_FILE, source=DATA_URL, extra={"dataset": "OpenStreetMap Romania (Geofabrik GeoPackage)"}
    )
    print(format_provenance_result(None, recorded=True))

## European Forest Disturbance Atlas (EFDA, Romania)

Downloads the Romania subset of the European Forest Disturbance Atlas, keeps the
annual disturbance stack (`annual_disturbances_1985_2023_romania.tif`) in
`data/raw/rasters/european_forest_disturbance_atlas/`, and discards the other
layers.

Source: Viana-Soto, A. and Senf, C. (2024), European Forest Disturbance Atlas,
Zenodo, https://doi.org/10.5281/zenodo.13333034.

In [ ]:
# European Forest Disturbance Atlas (Romania). Downloads the archive, keeps only
# the annual disturbance stack used by the study, and removes everything else.
# Clipping to the AOI is done later, in the processing step.
import zipfile

import requests

from utils.paths import get_project_paths
from utils.provenance import (
    format_provenance_result,
    provenance_path_for,
    verify_provenance,
    write_provenance,
)

# Resolve paths from the repo root, so this works whatever folder the notebook runs from.
paths = get_project_paths()
OUT_DIR = paths.raw / "rasters" / "european_forest_disturbance_atlas"
ZIP_FILE = OUT_DIR / "romania.zip"
WANTED = "annual_disturbances_1985_2023_romania.tif"
OUT_FILE = OUT_DIR / WANTED

DATA_URL = "https://zenodo.org/records/13333034/files/romania.zip?download=1"

# Set to True to download and extract again even if the file already exists.
FORCE = False

OUT_DIR.mkdir(parents=True, exist_ok=True)

if OUT_FILE.exists() and not FORCE:
    print(f"Already present: {OUT_FILE.name}")
else:
    print(f"Downloading to {ZIP_FILE} ...")
    with requests.get(DATA_URL, stream=True, timeout=600) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        done = 0
        with open(ZIP_FILE, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                done += len(chunk)
                if total:
                    print(f"\r  {done/1e6:7.1f} / {total/1e6:7.1f} MB", end="")
        print()

    print(f"Extracting {WANTED} ...")
    with zipfile.ZipFile(ZIP_FILE) as z:
        member = next((m for m in z.namelist() if m.endswith(WANTED)), None)
        if member is None:
            available = ", ".join(z.namelist())
            raise RuntimeError(f"{WANTED} not found in archive. Contents: {available}")
        with z.open(member) as src, open(OUT_FILE, "wb") as dst:
            for chunk in iter(lambda: src.read(1 << 20), b""):
                dst.write(chunk)
    ZIP_FILE.unlink()
    print(f"Kept: {OUT_FILE.name}")

# Record provenance on first run, or verify against it on later runs.
if provenance_path_for(OUT_FILE).exists():
    print(format_provenance_result(verify_provenance(OUT_FILE)))
else:
    write_provenance(
        OUT_FILE, source=DATA_URL, extra={"dataset": "European Forest Disturbance Atlas (Romania)"}
    )
    print(format_provenance_result(None, recorded=True))

## CORINE Land Cover 2018 (Copernicus Land Monitoring Service)

Requests CORINE Land Cover 2018 (vector) for Romania as a GeoPackage
through the CLMS download API, then saves it to
`data/raw/vectors/corine_land_cover/`. The CLMS download is asynchronous, so the
submitted task ID is recorded in `data/cache/api/clms_tasks.json`; a re-run polls
that existing task before submitting a new one, and reuses any GeoPackage or
manually downloaded archive already present. Once polling commences
this cell can be stopped and restarted, and it will pick up where it left off.
The request is restricted to Romania by NUTS code (a vector bounding-box
request returns empty geometry, CLMS error DR-002).

Source: Copernicus Land Monitoring Service, CORINE Land Cover 2018,
https://land.copernicus.eu/en/products/corine-land-cover/clc2018.

In [ ]:
# CORINE Land Cover 2018 (CLMS API, Romania subset, vector GeoPackage).
# Uses any artifact already in the output directory (a manually downloaded .gpkg
# or CLMS .zip); otherwise authenticates, submits a NUTS-restricted request for
# Romania, polls until ready, and downloads. The CLMS download is asynchronous.
# Clipping to the AOI and selection of forest classes are done later, in processing.
# API documentation: https://eea.github.io/clms-api-docs/authentication.html

import json
import os
import time
import zipfile
from pathlib import Path

import jwt  # PyJWT
import requests
from dotenv import load_dotenv

from utils.paths import get_project_paths
from utils.provenance import (
    format_provenance_result,
    provenance_path_for,
    verify_provenance,
    write_provenance,
)

paths = get_project_paths()
OUT_DIR = paths.raw / "vectors" / "corine_land_cover"
OUT_FILE = OUT_DIR / "corine_clc2018_romania.gpkg"
SOURCE = "CLMS API: CORINE Land Cover 2018, NUTS RO"

# CORINE Land Cover 2018, vector product, from the CLMS dataset catalogue.
DATASET_UID = "0407d497d3c44bcd93ce8fd5bf78596a"
DOWNLOAD_INFO_ID = "1bda2fbd-3230-42ba-98cf-69c96ac063bc"
NUTS = "RO"  # Romania (NUTS 0); clipped to the AOI later, in processing.
OUTPUT_FORMAT = "GPKG"
OUTPUT_GCS = "EPSG:3035"

API = "https://land.copernicus.eu/api"
POLL_SECONDS = 30
MAX_POLLS = 500
FORCE = False

OUT_DIR.mkdir(parents=True, exist_ok=True)


def _keep_gpkg(zip_path: Path) -> None:
    """Extract the single GeoPackage from a CLMS archive to OUT_FILE, then delete the archive."""
    with zipfile.ZipFile(zip_path) as z:
        member = next((m for m in z.namelist() if m.lower().endswith(".gpkg")), None)
        if member is None:
            raise RuntimeError(f"No .gpkg in archive. Contents: {z.namelist()}")
        with z.open(member) as src, open(OUT_FILE, "wb") as dst:
            for chunk in iter(lambda: src.read(1 << 20), b""):
                dst.write(chunk)
    zip_path.unlink()


# Resolve the GeoPackage: reuse an existing one, adopt a manual download
# (.gpkg or .zip already in OUT_DIR), or fetch it from the CLMS API.
existing = sorted(OUT_DIR.glob("*.gpkg"))
manual_zips = sorted(OUT_DIR.glob("*.zip"))

if existing and not FORCE:
    if existing[0] != OUT_FILE:
        existing[0].rename(OUT_FILE)
    print(f"Present: {OUT_FILE.name}")
elif manual_zips and not FORCE:
    print(f"Adopting manual download: {manual_zips[0].name}")
    _keep_gpkg(manual_zips[0])
    print(f"Kept: {OUT_FILE.name}")
else:
    # 1. Authenticate: sign a short-lived JWT with the service key and exchange
    #    it for a bearer access token.
    load_dotenv(paths.repo_root / ".env")
    key_path = os.environ.get("CLMS_SERVICE_KEY_PATH")
    if not key_path:
        raise RuntimeError("Set CLMS_SERVICE_KEY_PATH in your .env file (see env.example).")
    key = json.loads(Path(key_path).read_text())
    now = int(time.time())
    grant = jwt.encode(
        {
            "iss": key["client_id"],
            "sub": key["user_id"],
            "aud": key["token_uri"],
            "iat": now,
            "exp": now + 3600,
        },
        key["private_key"],
        algorithm="RS256",
    )
    token_resp = requests.post(
        key["token_uri"],
        data={"grant_type": "urn:ietf:params:oauth:grant-type:jwt-bearer", "assertion": grant},
        headers={"Accept": "application/json"},
        timeout=60,
    )
    token_resp.raise_for_status()
    auth = {
        "Authorization": f"Bearer {token_resp.json()['access_token']}",
        "Accept": "application/json",
    }

    # 2. Submit the request, restricted to Romania (NUTS 0). A vector bounding-box
    #    clip of CLC returns an empty geometry on this AOI (CLMS error DR-002), so
    #    the request is made by NUTS and clipped to the AOI later, in processing.
    body = {
        "Datasets": [
            {
                "DatasetID": DATASET_UID,
                "DatasetDownloadInformationID": DOWNLOAD_INFO_ID,
                "NUTS": NUTS,
                "OutputFormat": OUTPUT_FORMAT,
                "OutputGCS": OUTPUT_GCS,
            }
        ]
    }
    submit = requests.post(
        f"{API}/@datarequest_post",
        headers={**auth, "Content-Type": "application/json"},
        json=body,
        timeout=60,
    )
    submit.raise_for_status()
    task_id = submit.json()["TaskIds"][0]["TaskID"]
    print(f"Submitted request, task {task_id}. Waiting for it to be prepared ...")

    # 3. Poll until the task reports a download URL, failing fast if it is
    #    rejected. The full task entry prints on the first poll so the response
    #    fields are visible.
    download_url = None
    for attempt in range(MAX_POLLS):
        time.sleep(POLL_SECONDS)
        status = requests.get(f"{API}/@datarequest_search", headers=auth, timeout=60)
        status.raise_for_status()
        entry = status.json().get(str(task_id), {})
        if attempt == 0:
            print(f"  status fields: {json.dumps(entry, indent=2)[:1000]}")
        entry_status = str(entry.get("Status", "")).lower()
        message = entry.get("Message") or entry.get("message") or ""
        if "reject" in entry_status or "fail" in entry_status or "code:" in str(message).lower():
            raise RuntimeError(f"CLMS rejected task {task_id}: {message or entry}")
        download_url = next(
            (
                v
                for k, v in entry.items()
                if isinstance(v, str) and v.startswith("http") and "download" in k.lower()
            ),
            None,
        )
        print(f"  poll {attempt + 1}: {'ready' if download_url else 'still preparing'}")
        if download_url:
            break
    if download_url is None:
        raise TimeoutError(
            f"Task {task_id} not ready after {MAX_POLLS} polls; inspect the status fields above."
        )

    # 4. Download the prepared archive and keep the GeoPackage.
    zip_file = OUT_DIR / "corine_clc2018_romania.zip"
    print(f"Downloading to {zip_file} ...")
    with requests.get(download_url, headers=auth, stream=True, timeout=600) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        done = 0
        with open(zip_file, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                done += len(chunk)
                if total:
                    print(f"\r  {done/1e6:7.1f} / {total/1e6:7.1f} MB", end="")
        print()
    _keep_gpkg(zip_file)
    print(f"Kept: {OUT_FILE.name}")

# Record provenance on first run, or verify against it on later runs.
if provenance_path_for(OUT_FILE).exists():
    print(format_provenance_result(verify_provenance(OUT_FILE)))
else:
    write_provenance(OUT_FILE, source=SOURCE, extra={"dataset": "CORINE Land Cover 2018 (Romania)"})
    print(format_provenance_result(None, recorded=True))

## FABDEM (Forest And Buildings removed Copernicus DEM, V1-2)

Downloads the two 1°x1° FABDEM tiles covering the AOI (`N45E024`, `N45E025`)
to `data/raw/rasters/fabdem/`, by fetching the 10°x10° archive that contains
them and extracting only those tiles.

Source: Neal, J. and Hawker, L. (2023), FABDEM V1-2, University of Bristol,
https://doi.org/10.5523/bris.s5hqmjcdj8yo2ibzi9b4ew3sn.

In [ ]:
# FABDEM V1-2 (Forest And Buildings removed Copernicus DEM). Downloads the 10-degree
# archive containing the two AOI tiles, extracts only those tiles, and removes the
# archive. Mosaicking, reprojection and AOI clipping are done later, in processing.
import zipfile

import requests

from utils.paths import get_project_paths
from utils.provenance import (
    format_provenance_result,
    provenance_path_for,
    verify_provenance,
    write_provenance,
)

paths = get_project_paths()
OUT_DIR = paths.raw / "rasters" / "fabdem"
ZIP_FILE = OUT_DIR / "N40E020-N50E030_FABDEM_V1-2.zip"

# The two 1-degree tiles (south-west corner naming) covering the AOI, both inside
# the N40E020-N50E030 block.
WANTED = ("N45E024_FABDEM_V1-2.tif", "N45E025_FABDEM_V1-2.tif")

# Direct download of the 10-degree archive from the University of Bristol repository.
DATA_URL = (
    "https://data.bris.ac.uk/datasets/s5hqmjcdj8yo2ibzi9b4ew3sn/N40E020-N50E030_FABDEM_V1-2.zip"
)

# Set to True to download and extract again even if the tiles already exist.
FORCE = False

OUT_DIR.mkdir(parents=True, exist_ok=True)

out_files = [OUT_DIR / name for name in WANTED]
if all(f.exists() for f in out_files) and not FORCE:
    print(f"Already present: {', '.join(f.name for f in out_files)}")
else:
    print(f"Downloading to {ZIP_FILE} ...")
    with requests.get(DATA_URL, stream=True, timeout=600) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        done = 0
        with open(ZIP_FILE, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                done += len(chunk)
                if total:
                    print(f"\r  {done/1e6:7.1f} / {total/1e6:7.1f} MB", end="")
        print()

    print(f"Extracting {len(WANTED)} tile(s) ...")
    with zipfile.ZipFile(ZIP_FILE) as z:
        names = z.namelist()
        for name in WANTED:
            member = next((m for m in names if m.endswith(name)), None)
            if member is None:
                raise RuntimeError(f"{name} not found in archive. Contents: {names}")
            with z.open(member) as src, open(OUT_DIR / name, "wb") as dst:
                for chunk in iter(lambda: src.read(1 << 20), b""):
                    dst.write(chunk)
    ZIP_FILE.unlink()
    print(f"Kept: {', '.join(f.name for f in out_files)}")

# Record provenance per tile on first run, or verify against it on later runs.
for out_file in out_files:
    if provenance_path_for(out_file).exists():
        print(format_provenance_result(verify_provenance(out_file)))
    else:
        write_provenance(out_file, source=DATA_URL, extra={"dataset": "FABDEM V1-2"})
        print(format_provenance_result(None, recorded=True))

## ESA WorldCover annual composites (2020)

Downloads the Sentinel-1 and Sentinel-2 annual composite tiles covering the AOI
(`N45E024`, `N45E025`) to `data/raw/rasters/worldcover_composites/`. The tile AWS S3
keys are resolved from the official WorldCover composite grid, so they are not
hard-coded. Four products are fetched per tile: Sentinel-2 RGBNIR and SWIR
median composites, the Sentinel-2 NDVI percentiles composite, and the
Sentinel-1 VV/VH GAMMA0 composite.

Source: ESA WorldCover project / VITO. Zanaga, D. et al. (2021), ESA WorldCover
10 m 2020 v100 (https://doi.org/10.5281/zenodo.5571936); annual composites via
the AWS Open Data Registry (https://registry.opendata.aws/esa-worldcover-vito-composites/).

In [ ]:
# ESA WorldCover annual composites (2020). Resolves the S3 tile keys for the AOI
# from the official composite grid, then downloads the four per-tile products
# (S2 RGBNIR, S2 NDVI percentiles, S2 SWIR, S1 VV/VH) to data/raw/, unmodified.
# Mosaicking, reprojection and AOI clipping are done later, in processing.
from urllib.parse import urlparse

import geopandas as gpd
import requests

from utils.paths import get_project_paths
from utils.provenance import (
    format_provenance_result,
    provenance_path_for,
    verify_provenance,
    write_provenance,
)

paths = get_project_paths()
OUT_DIR = paths.raw / "rasters" / "worldcover_composites"
AOI_PATH = paths.aoi

# Official WorldCover composite tile grid (FlatGeobuf), with one S3 path column
# per product and year.
GRID_URL = "https://esa-worldcover.s3.eu-central-1.amazonaws.com/esa_worldcover_grid_composites.fgb"

# Products to fetch for 2020 (grid column -> S3 region endpoint host suffix).
PRODUCT_COLUMNS = (
    "s2_rgbnir_2020",
    "s2_ndvi_2020",
    "s2_swir_2020",
    "s1_vvvhratio_2020",
)

# Set to True to download again even if a tile already exists.
FORCE = False

if not AOI_PATH.exists():
    raise FileNotFoundError(
        f"Required AOI not found: {AOI_PATH}\n"
        "WorldCover tiles are selected by intersection with the AOI. Download "
        "Natura 2000 (above) and run the AOI-generation step in notebooks/002, "
        "then re-run this cell."
    )

OUT_DIR.mkdir(parents=True, exist_ok=True)


def _https(s3_path: str) -> str:
    """Convert an s3://bucket/key path to its eu-central-1 HTTPS URL."""
    parsed = urlparse(s3_path)
    return f"https://{parsed.netloc}.s3.eu-central-1.amazonaws.com/{parsed.path.lstrip('/')}"


# Resolve the AOI tile URLs from the grid (authoritative; no hard-coded keys).
aoi_geom = gpd.read_file(AOI_PATH).to_crs("EPSG:4326").union_all()
grid = gpd.read_file(GRID_URL)
hits = grid[grid.intersects(aoi_geom)]
print(f"Intersecting tiles: {hits['tile'].tolist()}")

urls = [
    _https(path)
    for col in PRODUCT_COLUMNS
    for path in hits[col].dropna()
    if isinstance(path, str) and path.startswith("s3://")
]

for url in urls:
    out_file = OUT_DIR / url.rsplit("/", 1)[1]
    if out_file.exists() and not FORCE:
        print(f"Already present: {out_file.name}")
    else:
        print(f"Downloading {out_file.name} ...")
        with requests.get(url, stream=True, timeout=600) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))
            done = 0
            with open(out_file, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)
                    done += len(chunk)
                    if total:
                        print(f"\r  {done/1e6:7.1f} / {total/1e6:7.1f} MB", end="")
            print()

    # Record provenance on first run, or verify against it on later runs.
    if provenance_path_for(out_file).exists():
        print(format_provenance_result(verify_provenance(out_file)))
    else:
        write_provenance(
            out_file, source=url, extra={"dataset": "ESA WorldCover annual composites 2020"}
        )
        print(format_provenance_result(None, recorded=True))

## Copernicus HR-VPP (Vegetation Phenology and Productivity, Season 1, 2020)
Downloads the 10 Season-1 VPP parameters for 2020 from the WEkEO Harmonised Data
Access (HDA) API, saving the GeoTIFFs to `data/raw/rasters/copernicus_vpp/`.
Requires a WEkEO account, with `WEKEO_USERNAME` and `WEKEO_PASSWORD` set in
`.env` (see README.md).

Source: Copernicus Land Monitoring Service, High Resolution Vegetation Phenology
and Productivity (HR-VPP), Season 1, 2020 (dataset EO:EEA:DAT:CLMS_HRVPP_VPP),
obtained via WEkEO HDA (https://www.wekeo.eu). Product information:
https://land.copernicus.eu/en/products/vegetation.

In [ ]:
# Fresh HR-VPP VPP download from WEkEO HDA.
#
# This cell:
#   1. Authenticates to WEkEO.
#   2. Searches the WEkEO HDA catalogue for exact HR-VPP VPP 2020 products.
#   3. Creates a WEkEO download order for each product.
#   4. Waits for the download to become ready using HEAD.
#   5. Streams the prepared HDA download endpoint.
#   6. Validates that the saved file is a real GeoTIFF.
#   7. Records provenance only after validation.
#
# Requires .env:
#   WEKEO_USERNAME=...
#   WEKEO_PASSWORD=...

import json
import os
import time
from pathlib import Path

import rasterio
import requests
from dotenv import load_dotenv
from tqdm.auto import tqdm

from utils.paths import get_project_paths
from utils.provenance import (
    format_provenance_result,
    provenance_path_for,
    verify_provenance,
    write_provenance,
)

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

paths = get_project_paths()

OUT_DIR = paths.raw / "rasters" / "copernicus_vpp"
DEBUG_DIR = paths.cache / "api" / "wekeo_debug"

OUT_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

SOURCE = "WEkEO HDA API: Copernicus HR-VPP VPP, Season 1, 2020, fresh download"

WEKEO_BASE = "https://gateway.prod.wekeo2.eu/hda-broker"
DATASET_ID = "EO:EEA:DAT:CLMS_HRVPP_VPP"

YEAR = 2020
PRODUCT_VERSION = "V101"
PRODUCT_GROUP_ID = "s1"

START = "2020-01-01T00:00:00.000Z"
END = "2021-01-01T00:00:00.000Z"

# AOI envelope from the CLMS diagnostic, in WEkEO bbox order [W, S, E, N].
BBOX = [
    24.25145263475321,
    45.31933538060352,
    25.231918895453145,
    45.73703207275138,
]

ALL_TILES = ["T34TGR", "T35TLL"]

ALL_PARAMETERS = [
    "AMPL",
    "EOSD",
    "EOSV",
    "LSLOPE",
    "MAXV",
    "MINV",
    "RSLOPE",
    "SOSD",
    "SOSV",
    "SPROD",
]

# Keep this True for the first run. After T34TGR AMPL validates, set False.
TEST_ONE_PRODUCT = False

TILES = ["T34TGR"] if TEST_ONE_PRODUCT else ALL_TILES
PARAMETERS = ["AMPL"] if TEST_ONE_PRODUCT else ALL_PARAMETERS

CHUNK_SIZE = 1 << 20
REQUEST_SLEEP_SECONDS = 1.0

DOWNLOAD_READY_TIMEOUT_SECONDS = 30 * 60
DOWNLOAD_READY_INITIAL_SLEEP_SECONDS = 2.0

CLEAN_TINY_OR_PARTIAL_FILES = True
TINY_FILE_THRESHOLD_BYTES = 1_000_000

SKIP_VALID_EXISTING = True


# ---------------------------------------------------------------------
# Utilities
# ---------------------------------------------------------------------


def _write_debug_json(name: str, obj) -> Path:
    path = DEBUG_DIR / name
    path.write_text(json.dumps(obj, indent=2, sort_keys=True, default=str) + "\n")
    return path


def _expected_id(tile: str, code: str) -> str:
    return f"VPP_{YEAR}_S2_{tile}-010m_{PRODUCT_VERSION}_{PRODUCT_GROUP_ID}_{code}"


def _expected_filename(tile: str, code: str) -> str:
    return f"{_expected_id(tile, code)}.tif"


def _is_tiff(path: Path) -> bool:
    if not path.exists() or path.stat().st_size < 4:
        return False

    with open(path, "rb") as f:
        magic = f.read(4)

    return magic in (b"II*\x00", b"MM\x00*")


def _validate_raster(path: Path, expected_size_bytes: int | None = None) -> dict:
    if not _is_tiff(path):
        preview = path.read_bytes()[:500] if path.exists() else b""
        raise RuntimeError(
            f"Not a TIFF: {path.name}; "
            f"size={path.stat().st_size if path.exists() else 0:,} bytes; "
            f"first bytes={preview!r}"
        )

    if expected_size_bytes is not None:
        got = path.stat().st_size
        minimum = int(expected_size_bytes * 0.90)

        if got < minimum:
            raise RuntimeError(
                f"File is much smaller than expected: {path.name}; "
                f"got={got:,} bytes, expected≈{expected_size_bytes:,} bytes"
            )

    with rasterio.open(path) as ds:
        return {
            "size_mb": round(path.stat().st_size / 1e6, 1),
            "width": ds.width,
            "height": ds.height,
            "count": ds.count,
            "dtype": ds.dtypes[0],
            "crs": str(ds.crs),
            "nodata": ds.nodata,
            "bounds": tuple(round(v, 3) for v in ds.bounds),
            "res": tuple(round(abs(v), 6) for v in ds.res),
        }


def _clean_bogus_outputs() -> None:
    if not CLEAN_TINY_OR_PARTIAL_FILES:
        return

    removed = []

    for path in sorted(OUT_DIR.glob("VPP_2020_S2_T*-010m_V101_s1_*.tif*")):
        is_partial = path.name.endswith(".part")
        is_tiny = path.is_file() and path.stat().st_size < TINY_FILE_THRESHOLD_BYTES

        if is_partial or is_tiny:
            removed.append((path.name, path.stat().st_size if path.exists() else 0))
            path.unlink(missing_ok=True)

            if path.suffix == ".tif":
                prov = provenance_path_for(path)
                if prov.exists():
                    prov.unlink()

    if removed:
        print("Removed bogus or partial files:")
        for name, size in removed:
            print(f"  {name}: {size:,} bytes")
    else:
        print("No bogus tiny or partial VPP files found.")


# ---------------------------------------------------------------------
# Authentication
# ---------------------------------------------------------------------


def _wekeo_token() -> str:
    load_dotenv(paths.repo_root / ".env")

    username = os.environ.get("WEKEO_USERNAME")
    password = os.environ.get("WEKEO_PASSWORD")

    if not username or not password:
        raise RuntimeError("Set WEKEO_USERNAME and WEKEO_PASSWORD in your .env file.")

    resp = requests.post(
        f"{WEKEO_BASE}/gettoken",
        headers={"Content-Type": "application/json"},
        json={"username": username, "password": password},
        timeout=60,
    )
    resp.raise_for_status()

    token = resp.json().get("access_token")

    if not token:
        raise RuntimeError(f"No access_token in WEkEO response: {resp.text[:1000]}")

    return token


def _headers(token: str) -> dict:
    return {
        "Authorization": f"Bearer {token}",
        "accept": "application/json",
    }


def _accept_terms(token: str) -> None:
    """Accept the CLMS data policy if required. Safe to rerun."""
    resp = requests.put(
        f"{WEKEO_BASE}/api/v1/termsaccepted/Copernicus_Land_Monitoring_Service_Data_Policy",
        headers=_headers(token),
        timeout=60,
    )

    if resp.status_code not in (200, 201, 204):
        print(f"Terms acceptance returned HTTP {resp.status_code}:")
        print(resp.text[:2000])
        resp.raise_for_status()


# ---------------------------------------------------------------------
# Search
# ---------------------------------------------------------------------


def _query_variants(code: str) -> list[dict]:
    """
    Search variants retained because WEkEO accepts both old flat fields and
    official HDA-style nested fields, but the flat form already returned the
    correct product IDs in this project.
    """
    flat_common = {
        "dataset_id": DATASET_ID,
        "start": START,
        "end": END,
        "bbox": BBOX,
    }

    official_common = {
        "datasetId": DATASET_ID,
        "boundingBoxValues": [
            {
                "name": "bbox",
                "bbox": BBOX,
            }
        ],
        "dateRangeSelectValues": [
            {
                "name": "temporal_interval",
                "start": START,
                "end": END,
            }
        ],
    }

    return [
        {
            **flat_common,
            "productType": code,
            "productGroupId": PRODUCT_GROUP_ID,
        },
        {
            **flat_common,
            "productType": [code],
            "productGroupId": PRODUCT_GROUP_ID,
        },
        {
            **flat_common,
            "productType": code,
        },
        {
            **official_common,
            "stringChoiceValues": [
                {"name": "productType", "value": code},
                {"name": "productGroupId", "value": PRODUCT_GROUP_ID},
            ],
        },
    ]


def _search_pages(token: str, base_query: dict, page_size: int = 200) -> list[dict]:
    features = []
    start_index = 0

    while True:
        query = {
            **base_query,
            "itemsPerPage": page_size,
            "startIndex": start_index,
        }

        resp = requests.post(
            f"{WEKEO_BASE}/api/v1/dataaccess/search",
            headers={**_headers(token), "Content-Type": "application/json"},
            json=query,
            timeout=120,
        )

        try:
            resp.raise_for_status()
        except requests.HTTPError:
            _write_debug_json(
                "search_http_error.json",
                {
                    "query": query,
                    "status_code": resp.status_code,
                    "text": resp.text,
                },
            )
            raise

        data = resp.json()
        page_features = data.get("features", [])
        total = int(data.get("properties", {}).get("totalResults", 0))

        features.extend(page_features)

        if not page_features or len(features) >= total:
            break

        start_index += len(page_features)

    return features


def _search_product(token: str, tile: str, code: str) -> dict:
    expected = _expected_id(tile, code)
    debug = []

    for variant_i, query in enumerate(_query_variants(code), start=1):
        features = _search_pages(token, query)
        ids = [f.get("id") for f in features]

        _write_debug_json(
            f"search_{tile}_{code}_variant{variant_i}.json",
            {
                "expected_id": expected,
                "query": query,
                "n_features": len(features),
                "ids": ids,
                "features": features,
            },
        )

        debug.append(
            {
                "variant": variant_i,
                "n_features": len(features),
                "sample_ids": ids[:20],
            }
        )

        print(f"{tile} {code}: variant {variant_i} returned {len(features)} features.")

        exact = [f for f in features if f.get("id") == expected]
        if exact:
            return exact[0]

    _write_debug_json(
        f"search_failed_{tile}_{code}.json",
        {
            "expected_id": expected,
            "debug": debug,
        },
    )

    raise RuntimeError(f"No exact WEkEO product match for {expected}. See {DEBUG_DIR}.")


# ---------------------------------------------------------------------
# HDA download order, wait, and stream
# ---------------------------------------------------------------------


def _create_download_order(token: str, feature: dict) -> str:
    props = feature.get("properties", {})

    body = {
        "cacheable": True,
        "dataset_id": DATASET_ID,
        "product_id": feature["id"],
        "location": props["location"],
    }

    resp = requests.post(
        f"{WEKEO_BASE}/api/v1/dataaccess/download",
        headers={**_headers(token), "Content-Type": "application/json"},
        json=body,
        timeout=120,
    )

    try:
        resp.raise_for_status()
    except requests.HTTPError:
        _write_debug_json(
            f"download_order_error_{feature['id']}.json",
            {
                "body": body,
                "status_code": resp.status_code,
                "text": resp.text,
            },
        )
        raise

    data = resp.json()
    _write_debug_json(f"download_order_{feature['id']}.json", data)

    download_id = data.get("download_id")

    if not download_id:
        raise RuntimeError(f"No download_id returned for {feature['id']}: {data}")

    return download_id


def _wait_for_download_ready(token: str, download_id: str) -> None:
    """
    Wait until the HDA download endpoint is ready.

    Expected:
      - 202 means the order is still being prepared.
      - 200 means the file should be streamable.
    """
    url = f"{WEKEO_BASE}/api/v1/dataaccess/download/{download_id}"

    started = time.time()
    sleep_seconds = DOWNLOAD_READY_INITIAL_SLEEP_SECONDS

    while True:
        resp = requests.head(
            url,
            headers=_headers(token),
            timeout=120,
            allow_redirects=False,
        )

        if resp.status_code == 200:
            return

        if resp.status_code == 202:
            elapsed = time.time() - started

            if elapsed > DOWNLOAD_READY_TIMEOUT_SECONDS:
                raise TimeoutError(
                    f"Timed out waiting for WEkEO download {download_id} "
                    f"after {DOWNLOAD_READY_TIMEOUT_SECONDS} seconds."
                )

            print(f"  download not ready yet; sleeping {sleep_seconds:.1f}s")
            time.sleep(sleep_seconds)
            sleep_seconds = min(sleep_seconds * 1.25, 30.0)
            continue

        _write_debug_json(
            f"download_head_error_{download_id}.json",
            {
                "status_code": resp.status_code,
                "headers": dict(resp.headers),
                "text": resp.text,
            },
        )

        raise RuntimeError(
            f"WEkEO download HEAD failed for {download_id}: "
            f"HTTP {resp.status_code}, {resp.text[:1000]}"
        )


def _stream_download(
    token: str,
    download_id: str,
    out_file: Path,
    expected_size_bytes: int | None = None,
) -> dict:
    """
    Stream the prepared HDA download endpoint.

    This deliberately does not follow or save a JSON Location response as a TIFF.
    """
    url = f"{WEKEO_BASE}/api/v1/dataaccess/download/{download_id}"
    tmp_file = out_file.with_suffix(out_file.suffix + ".part")

    with requests.get(
        url,
        headers=_headers(token),
        stream=True,
        timeout=1800,
        allow_redirects=True,
    ) as resp:
        resp.raise_for_status()

        content_type = resp.headers.get("content-type", "").lower()
        content_length = int(resp.headers.get("content-length", 0))

        if "application/json" in content_type:
            text = resp.text
            _write_debug_json(
                f"download_json_instead_of_tif_{download_id}.json",
                {
                    "status_code": resp.status_code,
                    "headers": dict(resp.headers),
                    "text": text,
                },
            )

            raise RuntimeError(
                f"WEkEO returned JSON instead of file bytes for {download_id}: " f"{text[:1000]}"
            )

        total = content_length or expected_size_bytes or 0
        done = 0

        with open(tmp_file, "wb") as f:
            for chunk in resp.iter_content(chunk_size=CHUNK_SIZE):
                if not chunk:
                    continue

                f.write(chunk)
                done += len(chunk)

                if total:
                    print(f"\r  {done / 1e6:8.1f} / {total / 1e6:8.1f} MB", end="")
                else:
                    print(f"\r  {done / 1e6:8.1f} MB", end="")

        print()

    diagnostics = _validate_raster(tmp_file, expected_size_bytes=expected_size_bytes)
    tmp_file.replace(out_file)

    return diagnostics


def _download_feature(token: str, feature: dict, out_file: Path) -> dict:
    props = feature.get("properties", {})
    expected_size = int(props.get("size", 0) or 0) or None

    download_id = _create_download_order(token, feature)
    print(f"  download_id: {download_id}")

    _wait_for_download_ready(token, download_id)

    return _stream_download(
        token=token,
        download_id=download_id,
        out_file=out_file,
        expected_size_bytes=expected_size,
    )


# ---------------------------------------------------------------------
# Provenance
# ---------------------------------------------------------------------


def _record_or_verify_provenance(path: Path) -> None:
    if provenance_path_for(path).exists():
        print(f"{path.name}: {format_provenance_result(verify_provenance(path))}")
        return

    code = path.stem.split("_s1_")[-1]
    tile = path.name.split("_")[3].split("-")[0]

    write_provenance(
        path,
        source=SOURCE,
        extra={
            "dataset": DATASET_ID,
            "year": YEAR,
            "season": PRODUCT_GROUP_ID,
            "tile": tile,
            "parameter": code,
            "product_version": PRODUCT_VERSION,
            "fresh_download": True,
            "api": "WEkEO HDA",
        },
    )

    print(f"{path.name}: {format_provenance_result(None, recorded=True)}")


# ---------------------------------------------------------------------
# Run
# ---------------------------------------------------------------------

_clean_bogus_outputs()

token = _wekeo_token()
_accept_terms(token)

print("Authenticated to WEkEO.")
print(f"Output directory: {OUT_DIR}")
print(f"Dataset: {DATASET_ID}")
print(f"BBOX [W, S, E, N]: {[round(v, 6) for v in BBOX]}")
print(f"Tiles: {', '.join(TILES)}")
print(f"Parameters: {', '.join(PARAMETERS)}")
print(f"TEST_ONE_PRODUCT: {TEST_ONE_PRODUCT}")

downloaded = []
skipped = []
failed = []

for tile in TILES:
    for code in tqdm(PARAMETERS, desc=f"Downloading {tile}"):
        out_file = OUT_DIR / _expected_filename(tile, code)

        try:
            feature = _search_product(token, tile, code)
            props = feature.get("properties", {})
            expected_size = int(props.get("size", 0) or 0) or None
            size_mb = (expected_size or 0) / 1e6

            print(f"\n{tile} {code}: found {feature['id']} ({size_mb:.1f} MB)")
            print(f"  source: {props.get('location', '')}")

            if SKIP_VALID_EXISTING and out_file.exists():
                diagnostics = _validate_raster(out_file, expected_size_bytes=expected_size)
                skipped.append(out_file)

                print(
                    f"  valid existing file, skipped: "
                    f"{diagnostics['size_mb']} MB, "
                    f"{diagnostics['width']}x{diagnostics['height']}, "
                    f"crs={diagnostics['crs']}"
                )
                continue

            diagnostics = _download_feature(token, feature, out_file)
            downloaded.append(out_file)

            print(
                f"  validated: {diagnostics['size_mb']} MB, "
                f"{diagnostics['width']}x{diagnostics['height']}, "
                f"bands={diagnostics['count']}, "
                f"dtype={diagnostics['dtype']}, "
                f"crs={diagnostics['crs']}, "
                f"nodata={diagnostics['nodata']}"
            )

            time.sleep(REQUEST_SLEEP_SECONDS)

        except Exception as exc:
            failed.append((tile, code, str(exc)))
            print(f"\nFAILED {tile} {code}: {exc}")

print()
print(f"Downloaded: {len(downloaded)}")
print(f"Skipped valid existing: {len(skipped)}")
print(f"Failed: {len(failed)}")

if failed:
    print("\nFailures:")
    for tile, code, message in failed:
        print(f"  {tile} {code}: {message}")
    print(f"\nDebug files: {DEBUG_DIR}")

print("\nProvenance:")
for out_file in sorted(OUT_DIR.glob("VPP_2020_S2_T*-010m_V101_s1_*.tif")):
    _record_or_verify_provenance(out_file)

print("\nDone.")

## AlphaEarth Foundations embeddings (2020)

Downloads the 64-band AlphaEarth Foundations annual satellite embeddings for 2020
over the AOI directly to `data/raw/rasters/alphaearth/`, via the Earth Engine
high-volume endpoint. The embeddings are read on their native UTM
grid at 10 m.

Source: Brown, Kazmierski, Pasquarella et al. (2025), AlphaEarth Foundations;
Google Satellite Embedding V1 Annual, Earth Engine collection
`GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL`.

Est. download time: 45 minutes (tested on a weekend with a very fast connection).

In [ ]:
## RUN ONCE: EE AUTHENTICATION
import ee

ee.Authenticate()

In [ ]:
# AlphaEarth Foundations embeddings (2020), via the Earth Engine high-volume
# endpoint and xee. Downloaded in AlphaEarth's native CRS (EPSG:32635, UTM 35N)
# at 10 m with no server-side reprojection. AlphaEarth's annual collection ships
# one image per Sentinel-2 tile, so the AOI's tiles are mosaicked in Earth Engine
# before streaming; otherwise xee retains only one tile per pixel and the rest
# is NaN. The export bbox is supplied in WGS84 (EE's native geometry CRS),
# computed from a 10 km-buffered AOI, giving a generous margin around the AOI
# regardless of any axis-rotation effects in EE's internal geometry handling.
# Reprojection to the project CRS (EPSG:3035) on the reference grid happens in
# notebook 002 at processing time, alongside the other raw rasters.
import os

import ee
import geopandas as gpd
import rioxarray
import xarray as xr
from dotenv import load_dotenv

from utils import terminology
from utils.paths import get_project_paths
from utils.provenance import (
    format_provenance_result,
    provenance_path_for,
    verify_provenance,
    write_provenance,
)

paths = get_project_paths()
OUT_DIR = paths.raw / "rasters" / "alphaearth"
OUT_FILE = OUT_DIR / "alphaearth_2020_aoi_32635.tif"

SOURCE = (
    "GEE high-volume endpoint: GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL, 2020, "
    "EE-side mosaic, exported in native EPSG:32635 at 10 m, WGS84 bbox from "
    "the 10 km-buffered AOI."
)
COLLECTION = "GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL"
YEAR_START, YEAR_END = "2020-01-01", "2021-01-01"
NATIVE_CRS = "EPSG:32635"  # AlphaEarth's native UTM zone over Făgăraș
PROJECT_CRS = terminology.CRS  # EPSG:3035, used only for buffering
SCALE_M = terminology.REF_RESOLUTION_M  # 10
BUFFER_M = 10_000.0  # 10 km, generous margin
HIGH_VOLUME = "https://earthengine-highvolume.googleapis.com"

OUT_DIR.mkdir(parents=True, exist_ok=True)

if OUT_FILE.exists():
    print(f"Present: {OUT_FILE.name}")
else:
    load_dotenv(paths.repo_root / ".env")
    project = os.environ.get("EE_PROJECT")
    if not project:
        raise RuntimeError("Set EE_PROJECT in your .env file (your Earth Engine Cloud project).")
    ee.Initialize(project=project, opt_url=HIGH_VOLUME)

    # Buffer in EPSG:3035 (true metres) then take the WGS84 bbox of the buffered
    # polygon. EE indexes and reasons in WGS84, so a WGS84 rectangle is what its
    # filter and export-window logic use directly, avoiding any implicit CRS
    # round trips that could shrink the export footprint.
    aoi_buffered_3035 = (
        gpd.read_file(paths.aoi).to_crs(PROJECT_CRS).geometry.iloc[0].buffer(BUFFER_M)
    )
    w, s, e, n = (
        gpd.GeoSeries([aoi_buffered_3035], crs=PROJECT_CRS).to_crs("EPSG:4326").total_bounds
    )
    aoi_export = ee.Geometry.Rectangle([w, s, e, n])  # WGS84, EE's native CRS

    tiles = ee.ImageCollection(COLLECTION).filterDate(YEAR_START, YEAR_END).filterBounds(aoi_export)
    n_tiles = int(tiles.size().getInfo())
    zones = tiles.aggregate_array("UTM_ZONE").distinct().getInfo()
    print(f"AlphaEarth 2020 tiles intersecting the AOI: {n_tiles}; UTM zones: {zones}")
    if n_tiles == 0:
        raise RuntimeError("No AlphaEarth tiles match the AOI and date filter.")
    if zones != ["35N"]:
        raise RuntimeError(
            f"Expected all tiles in UTM zone 35N for Făgăraș; got {zones}. "
            f"Cell assumes single-zone export; revise if AOI spans multiple zones."
        )

    # Mosaic on the EE side, then stream the mosaic in its native CRS.
    mosaic_collection = ee.ImageCollection([tiles.mosaic()])

    print(f"Streaming the mosaic in {NATIVE_CRS} at {SCALE_M} m (10 km AOI buffer) ...")
    ds = xr.open_dataset(
        mosaic_collection,
        engine="ee",
        crs=NATIVE_CRS,
        scale=SCALE_M,
        geometry=aoi_export,
    )
    # Rename X/Y to x/y up-front so rioxarray auto-detects the spatial dims and
    # the binding survives the subsequent .astype copy.
    da = (
        ds.isel(time=0)
        .to_array(dim="band")
        .transpose("band", "Y", "X")
        .rename({"X": "x", "Y": "y"})
        .astype("float32")
        .rio.write_crs(NATIVE_CRS)
    )
    # Force north-up affine in case xee returns y ascending.
    if float(da.y.values[0]) < float(da.y.values[-1]):
        da = da.sortby("y", ascending=False).rio.write_crs(NATIVE_CRS)

    tmp_file = OUT_FILE.with_suffix(".tif.tmp")
    print(
        f"Writing {OUT_FILE.name} ({da.sizes['band']} bands, "
        f"{da.sizes['y']}x{da.sizes['x']} px) ..."
    )
    da.rio.to_raster(tmp_file, driver="GTiff", tiled=True, compress="DEFLATE", BIGTIFF="IF_SAFER")
    tmp_file.rename(OUT_FILE)
    print("Done.")

if provenance_path_for(OUT_FILE).exists():
    print(format_provenance_result(verify_provenance(OUT_FILE)))
else:
    write_provenance(OUT_FILE, source=SOURCE, extra={"dataset": "AlphaEarth Foundations (2020)"})
    print(format_provenance_result(None, recorded=True))

## TESSERA embeddings (2020)

Downloads the 128-band TESSERA annual embedding for the AOI to
`data/raw/rasters/tessera/`, one GeoTIFF per 0.1-degree tile on its native UTM
grid. `TESSERA_VERSION` in the cell selects the embedding version, `v1.1` (the
tiles used in the study) or `v2`; nothing downstream sees the version. Changing
it moves everything the installed version produced (tiles, mosaic, feature
stack, caches, results and figures) to `archive/tessera_<version>/` and brings
the requested version's back, so switching and rolling back take seconds; the
steps the requested version has not run yet are re-run from notebook 002 on.

Source: Tessera foundation model, University of Cambridge Earth Observation
Group (https://github.com/ucam-eo/geotessera); tiles fetched from the Source
Cooperative mirror (https://source.coop/tessera/tessera).

In [ ]:
# TESSERA embeddings (2020). TESSERA_VERSION is the only place in the repository
# where the embedding version is chosen: the tiles land in data/raw/rasters/tessera/
# under version-free names and every later notebook and script reads that directory
# as "tessera". The tiles come straight from the Source Cooperative mirror through
# scripts/download_tessera_tiles.py, because the geotessera client's registry can
# lag the object store: each 0.1-degree tile's quantised array and per-pixel scales
# are cached under data/cache/tessera/ and exported as one 128-band float32 GeoTIFF
# on its native UTM grid, as `geotessera download --format tiff` writes it (a
# fresh v1.1 export is pixel-identical to the study's original tiles). Changing
# the version moves everything the installed version produced (tiles, mosaic,
# feature stack, caches, results and figures; see utils/tessera_version.py) to
# archive/tessera_<version>/ and brings the requested version's back from there,
# so switching, and rolling back, is a rename and takes seconds; whatever the
# requested version has not produced yet is rebuilt by re-running the pipeline.
# Reprojection and AOI clipping are done in notebook 002.

import math
import sys

import geopandas as gpd

from utils.paths import get_project_paths
from utils.provenance import (
    format_provenance_result,
    provenance_path_for,
    verify_provenance,
    write_provenance,
)
from utils.tessera_version import DATASET_PATHS, dataset_label, installed_version, switch_version

paths = get_project_paths()
if str(paths.repo_root) not in sys.path:
    sys.path.insert(0, str(paths.repo_root))
from scripts.download_tessera_tiles import fetch_tiles

TESSERA_VERSION = "v2"  # "v1.1" (the study's original tiles) or "v2"; see DATASET_PATHS
YEAR = 2020
TILE_STEP_DEG = 0.1  # tiles are 0.1-degree cells, named by their centre

OUT_DIR = paths.raw / "rasters" / "tessera"
CACHE_DIR = paths.cache / "tessera"
DATASET = dataset_label(TESSERA_VERSION, YEAR)
SOURCE = (
    f"https://data.source.coop/tessera/tessera/npy/{DATASET_PATHS[TESSERA_VERSION]}/{YEAR}/ via "
    "scripts.download_tessera_tiles.fetch_tiles (int8 array x per-pixel scales, georeferenced "
    "from the landmask tile), AOI bounding box"
)

installed = installed_version(paths.repo_root)
if installed != TESSERA_VERSION:
    moved = switch_version(paths.repo_root, installed, TESSERA_VERSION)
    print(
        f"Switched TESSERA {installed or 'none'} -> {TESSERA_VERSION}: parked {moved['parked']} "
        f"path(s) under archive/, restored {moved['restored']}."
    )
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Every tile whose cell intersects the AOI's lon/lat bounding box.
w, s, e, n = map(float, gpd.read_file(paths.aoi).to_crs("EPSG:4326").total_bounds)
names = sorted(
    f"grid_{(i + 0.5) * TILE_STEP_DEG:.2f}_{(j + 0.5) * TILE_STEP_DEG:.2f}"
    for i in range(math.floor(w / TILE_STEP_DEG), math.floor(e / TILE_STEP_DEG) + 1)
    for j in range(math.floor(s / TILE_STEP_DEG), math.floor(n / TILE_STEP_DEG) + 1)
)
print(f"Tiles in AOI: {len(names)}")

missing = [name for name in names if not (OUT_DIR / f"{name}_{YEAR}.tiff").is_file()]
if not missing:
    print("All tiles already present; skipping download.")
else:
    print(f"Fetching {len(missing)} TESSERA {TESSERA_VERSION} tile(s) ...")
    fetch_tiles(
        missing, dataset_version=TESSERA_VERSION, year=YEAR, cache_dir=CACHE_DIR, tiff_dir=OUT_DIR
    )
    failed = [name for name in missing if not (OUT_DIR / f"{name}_{YEAR}.tiff").is_file()]
    if failed:
        raise RuntimeError(f"{len(failed)} tile(s) not downloaded: {', '.join(failed)}")
    print("Done.")

# Record provenance per GeoTIFF on first run, or verify against it on later runs.
for name in names:
    out_file = OUT_DIR / f"{name}_{YEAR}.tiff"
    if provenance_path_for(out_file).exists():
        print(f"{out_file.name}: {format_provenance_result(verify_provenance(out_file))}")
    else:
        write_provenance(out_file, source=SOURCE, extra={"dataset": DATASET})
        print(f"{out_file.name}: {format_provenance_result(None, recorded=True)}")

## Sabatini (2020) primary forest probability map (manual download)

The iDiv data repository (dataset 1841) requires a browser session to download;
direct programmatic download is not supported. Download the following three files
manually from https://idata.idiv.de/ddm/Data/ShowData/1841 and place them in
`data/raw/existing_products/sabatini/`:

- `1841_2_pred_ogbrt_ssbtc5_lr02_March20.tif` — BRT probability prediction (used for evaluation)
- `1841_2_primary_composite_2019120613.tif` — input composite (context)
- `1841_2_Forests_and_CO_Metadata1.xlsx` — metadata workbook

Then run the cell below to record provenance.

In [ ]:
# Sabatini (2020) primary-forest products (iDiv dataset 1841).
# Files must be placed manually in data/raw/existing_products/sabatini/
# (the iDiv repository requires a browser session; direct download is not supported).
# Run this cell after placing the files to record or verify provenance.
from utils.paths import get_project_paths
from utils.provenance import (
    format_provenance_result,
    provenance_path_for,
    verify_provenance,
    write_provenance,
)

paths = get_project_paths()
OUT_DIR = paths.raw / "existing_products" / "sabatini"

EXPECTED = (
    "1841_22_1841_2_pred_ogbrt_ssbtc5_lr02_March20.tif",
    "1841_22_1841_2_primary_composite_2019120613.tif",
    "1841_22_1841_2_Forests_and_CO_Metadata1.xlsx",
)
SOURCE_PAGE = "https://idata.idiv.de/ddm/Data/ShowData/1841"

for name in EXPECTED:
    out_file = OUT_DIR / name
    if not out_file.exists():
        print(f"MISSING: {name} — place in {OUT_DIR} and re-run.")
        continue
    if provenance_path_for(out_file).exists():
        print(f"{out_file.name}: {format_provenance_result(verify_provenance(out_file))}")
    else:
        write_provenance(
            out_file,
            source=SOURCE_PAGE,
            extra={"dataset": "Sabatini (2020) primary forest map"},
        )
        print(f"{out_file.name}: {format_provenance_result(None, recorded=True)}")

## Munteanu et al. (2022) forest structure and HCVF maps

Downloads the Romania HCVF and forest-structure products to
`data/raw/existing_products/munteanu/`, keeping the GeoTIFF layers and the
readme.

Source: Munteanu, C. et al. (2022), data on Zenodo,
https://doi.org/10.5281/zenodo.4555708.

In [ ]:
# Munteanu et al. (2022) Romania HCVF / forest-structure products (Zenodo).
# Downloads the archive, keeps the GeoTIFF layers and the readme, and discards
# the regenerable .aux.xml / .ovr / .xml sidecars. Clipping is done later.
import zipfile
from pathlib import Path

import requests

from utils.paths import get_project_paths
from utils.provenance import (
    format_provenance_result,
    provenance_path_for,
    verify_provenance,
    write_provenance,
)

paths = get_project_paths()
OUT_DIR = paths.raw / "existing_products" / "munteanu"
ZIP_FILE = OUT_DIR / "Romania_HCVF_data.zip"

DATA_URL = "https://zenodo.org/records/4555708/files/Romania_HCVF_data.zip?download=1"
SOURCE = "https://doi.org/10.5281/zenodo.4555708"

FORCE = False
OUT_DIR.mkdir(parents=True, exist_ok=True)

existing = sorted(OUT_DIR.glob("*.tif"))
if existing and not FORCE:
    print(f"Already present: {', '.join(f.name for f in existing)}")
else:
    print(f"Downloading to {ZIP_FILE} ...")
    with requests.get(DATA_URL, stream=True, timeout=600) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        done = 0
        with open(ZIP_FILE, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                done += len(chunk)
                if total:
                    print(f"\r  {done/1e6:7.1f} / {total/1e6:7.1f} MB", end="")
        print()

    print("Extracting GeoTIFFs and readme ...")
    with zipfile.ZipFile(ZIP_FILE) as z:
        wanted = [
            m
            for m in z.namelist()
            if m.lower().endswith(".tif") or m.lower().endswith("readme.txt")
        ]
        if not wanted:
            raise RuntimeError(f"No GeoTIFFs found in archive. Contents: {z.namelist()}")
        for member in wanted:
            out_file = OUT_DIR / Path(member).name
            with z.open(member) as src, open(out_file, "wb") as dst:
                for chunk in iter(lambda: src.read(1 << 20), b""):
                    dst.write(chunk)
    ZIP_FILE.unlink()
    print(f"Kept: {', '.join(f.name for f in sorted(OUT_DIR.glob('*.tif')))}")

for out_file in sorted(OUT_DIR.glob("*")):
    if out_file.name.endswith(".provenance.json"):
        continue
    if provenance_path_for(out_file).exists():
        print(f"{out_file.name}: {format_provenance_result(verify_provenance(out_file))}")
    else:
        write_provenance(
            out_file,
            source=SOURCE,
            extra={"dataset": "Munteanu et al. (2022) HCVF and forest structure"},
        )
        print(f"{out_file.name}: {format_provenance_result(None, recorded=True)}")

## Kathmann et al. (2017) potential primary forests of Romania (Greenpeace)

Downloads the Greenpeace "Potential Primary Forests Map of Romania" vector data
(KMZ and shapefiles, English release) to `data/raw/existing_products/kathmann/`,
extracting the shapefile components and discarding the KMZ.

Source: Kathmann, F., Ciutea, A., Biriș, I.-A., Ibisch, P.L. and Sălăgeanu, V.
(2017), Potential Primary Forests Map of Romania (Greenpeace CEE Romania;
Eberswalde University; A.I. Cuza University of Iași),
https://doi.org/10.13140/RG.2.2.36773.60644. Data via Greenpeace Romania:
https://www.greenpeace.org/romania/raport/1235/.

In [ ]:
# Kathmann et al. (2017) potential primary forests of Romania (Greenpeace KMZ +
# shapefiles, English release). Downloads the archive, keeps the shapefile
# components, and discards the KMZ. Reprojection and AOI clipping are done later.
import zipfile
from pathlib import Path

import requests

from utils.paths import get_project_paths
from utils.provenance import (
    format_provenance_result,
    provenance_path_for,
    verify_provenance,
    write_provenance,
)

paths = get_project_paths()
OUT_DIR = paths.raw / "existing_products" / "kathmann"
ZIP_FILE = OUT_DIR / "harta-padurile-virgine-en-kmz-si-shapefile.zip"

DATA_URL = (
    "https://www.greenpeace.org/static/planet4-romania-stateless/2020/11/"
    "4f4fa4eb-harta-padurile-virgine-en-kmz-si-shapefile.zip"
)
SOURCE_PAGE = "https://www.greenpeace.org/romania/raport/1235/"

# Shapefile sidecar extensions to keep; the KMZ is discarded.
SHP_EXTS = (".shp", ".shx", ".dbf", ".prj", ".cpg", ".sbn", ".sbx")

FORCE = False
OUT_DIR.mkdir(parents=True, exist_ok=True)

existing = sorted(OUT_DIR.glob("*.shp"))
if existing and not FORCE:
    print(f"Already present: {', '.join(f.name for f in existing)}")
else:
    print(f"Downloading to {ZIP_FILE} ...")
    with requests.get(DATA_URL, stream=True, timeout=600) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        done = 0
        with open(ZIP_FILE, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                done += len(chunk)
                if total:
                    print(f"\r  {done/1e6:7.1f} / {total/1e6:7.1f} MB", end="")
        print()

    print("Extracting shapefile components ...")
    with zipfile.ZipFile(ZIP_FILE) as z:
        wanted = [m for m in z.namelist() if m.lower().endswith(SHP_EXTS)]
        if not wanted:
            raise RuntimeError(f"No shapefile components in archive. Contents: {z.namelist()}")
        for member in wanted:
            out_file = OUT_DIR / Path(member).name
            with z.open(member) as src, open(out_file, "wb") as dst:
                for chunk in iter(lambda: src.read(1 << 20), b""):
                    dst.write(chunk)
    ZIP_FILE.unlink()
    print(f"Kept: {', '.join(f.name for f in sorted(OUT_DIR.glob('*.shp')))}")

# Record provenance per file on first run, or verify against it on later runs.
for out_file in sorted(OUT_DIR.glob("*")):
    if out_file.name.endswith(".provenance.json"):
        continue
    if provenance_path_for(out_file).exists():
        print(f"{out_file.name}: {format_provenance_result(verify_provenance(out_file))}")
    else:
        write_provenance(
            out_file,
            source=SOURCE_PAGE,
            extra={"dataset": "Kathmann et al. (2017) potential primary forests (Greenpeace)"},
        )
        print(f"{out_file.name}: {format_provenance_result(None, recorded=True)}")

## European mountain areas (EEA, 2008)

Downloads the EEA European mountain areas delineation to
`data/raw/vectors/european_mountain_areas/`, extracting the shapefile and
discarding any other contents.

Source: European Environment Agency / ETC-ULS (2008), European mountain areas
(version 1, Dec. 2008), reference layer for EEA Report No 6/2010,
https://www.eea.europa.eu/en/datahub/datahubitem-view/48b6f8a7-9e5a-4865-bd05-59c29348b5fb.

In [ ]:
# European mountain areas (EEA, 2008). Downloads the shapefile components from the
# EEA datashare folder. Selection of the Carpathian massif, reprojection and
# clipping are done later, in processing.
import requests

from utils.paths import get_project_paths
from utils.provenance import (
    format_provenance_result,
    provenance_path_for,
    verify_provenance,
    write_provenance,
)

paths = get_project_paths()
OUT_DIR = paths.raw / "vectors" / "european_mountain_areas"

BASE = "https://sdi.eea.europa.eu/datashare/s/wcYJyztMRoCz4Kq/download?path=%2F&files="
STEM = "m_massifs_v1"
EXTS = (".shp", ".shx", ".dbf", ".prj", ".sbn", ".sbx")
SOURCE_PAGE = (
    "https://www.eea.europa.eu/en/datahub/datahubitem-view/" "48b6f8a7-9e5a-4865-bd05-59c29348b5fb"
)

FORCE = False
OUT_DIR.mkdir(parents=True, exist_ok=True)

for ext in EXTS:
    name = STEM + ext
    out_file = OUT_DIR / name
    if out_file.exists() and not FORCE:
        print(f"Already present: {out_file.name}")
    else:
        print(f"Downloading {out_file.name} ...")
        with requests.get(BASE + name, stream=True, timeout=300) as r:
            r.raise_for_status()
            with open(out_file, "wb") as f:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    f.write(chunk)

    if provenance_path_for(out_file).exists():
        print(f"{out_file.name}: {format_provenance_result(verify_provenance(out_file))}")
    else:
        write_provenance(
            out_file, source=SOURCE_PAGE, extra={"dataset": "European mountain areas (EEA, 2008)"}
        )
        print(f"{out_file.name}: {format_provenance_result(None, recorded=True)}")